# ROBUST04 Hybrid-Optimized Pipeline (Target: MAP 0.35)

## Key Findings from Parameter Tuning:
1. **Weighted RRF > Pure RRF** (0.2738 vs 0.2728)
2. **Optimal k = 30** (not 60!)
3. **Optimal rerank_depth = 1000** (+3% MAP vs 500)
4. **Need additional signals** to reach 0.35

## This Notebook:
- Combines weighted RRF + optimized parameters
- Adds strategy to reach MAP 0.35
- Expected: ~0.28-0.32 MAP (baseline) → 0.35+ with enhancements

In [ ]:
hugging_face_1bLLamaInstruct = "WRITE_YOUR_HF_TOKEN_HERE"

In [ ]:
from huggingface_hub import login

login(hugging_face_1bLLamaInstruct)

In [ ]:
# ============================================================================
# JAVA 21 SETUP
# ============================================================================

import os
import subprocess

print("Checking Java version...")

try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 ready!")
print("="*70)

In [ ]:
# ============================================================================
# INSTALL PACKAGES
# ============================================================================

print("Installing packages...")

!pip install -q transformers>=4.46.0
!pip install -q pyserini==0.22.1
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy
!pip install faiss-cpu --no-cache

print("✓ All packages installed!")

In [ ]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings

warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive
import os

try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False

if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'
    
    if os.path.exists(BASE_DIR):
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {BASE_DIR}")
    else:
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive',
        ]
        for path in search_paths:
            if os.path.exists(os.path.join(path, 'queriesROBUST.txt')):
                os.chdir(path)
                break
else:
    BASE_DIR = os.getcwd()

print(f"\nWorking directory: {os.getcwd()}")
if os.path.exists('queriesROBUST.txt'):
    print("✓ Found queriesROBUST.txt")
if os.path.exists('qrels_50_Queries'):
    print("✓ Found qrels_50_Queries")

## Configuration

In [ ]:
# OPTIMIZED CONFIGURATION BASED ON TUNING RESULTS
USE_MONOT5 = True
OPTIMAL_K = 30  # From tuning: k=30 > k=60
OPTIMAL_RERANK_DEPTH = 1000  # From tuning: depth=1000 best

# Weighted RRF (proven better than pure RRF)
W_BM25 = 1.0
W_RM3 = 1.5
W_MONOT5 = 2.0

print(f"\n🎯 HYBRID-OPTIMIZED Configuration:")
print(f"  • Fusion: Weighted RRF (W_BM25={W_BM25}, W_RM3={W_RM3}, W_MONOT5={W_MONOT5})")
print(f"  • RRF k: {OPTIMAL_K} (tuned)")
print(f"  • Rerank depth: {OPTIMAL_RERANK_DEPTH} (tuned)")
print(f"  • MonoT5 Reranker: {USE_MONOT5}")
print(f"\n📈 Expected Performance:")
print(f"  • Training MAP: ~0.28-0.29")
print(f"  • With additional signals: 0.30-0.35")

## Load Index and Searchers

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# BM25 searcher
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)

# RM3 searcher - OPTIMIZED PARAMETERS
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)
rm3_searcher.set_rm3(fb_docs=20, fb_terms=15, original_query_weight=0.6)

searchers = [bm25_searcher, rm3_searcher]

print("✓ Configured BM25 + RM3 (optimized)")

## Load Queries

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## Helper Functions

In [ ]:
def classify_query(query_text):
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## Multi-Method Retrieval

In [ ]:
def multi_method_retrieval(query_text, k=1000):
    bm25_hits = searchers[0].search(query_text, k=k)
    rm3_hits = searchers[1].search(query_text, k=k)

    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    rm3_results = [(h.docid, h.score) for h in rm3_hits]

    return bm25_results, rm3_results

## Load MonoT5 Reranker

In [ ]:
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5 Reranker...")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        mn = "castorini/monot5-base-msmarco-10k"
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  ✓ MonoT5 loaded successfully")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

## MonoT5 Reranker Function

In [ ]:
def rerank_monot5(query, doc_ids, batch_size=8):
    """MonoT5 reranking"""
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            doc = index_reader.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except:
            pass

    if not doc_texts:
        return None

    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()
                all_scores.extend(batch_scores)
        except Exception as e:
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## HYBRID-OPTIMIZED Pipeline

Combines:
- Weighted RRF (better than pure RRF)
- Optimal k=30 (from tuning)
- Optimal rerank_depth=1000 (from tuning)

In [ ]:
def hybrid_optimized_pipeline(query, rerank_depth=OPTIMAL_RERANK_DEPTH, k_rrf=OPTIMAL_K):
    """
    HYBRID-OPTIMIZED: Weighted RRF + Tuned Parameters
    
    Based on tuning results:
    - Weighted RRF outperforms pure RRF
    - k=30 is optimal (not 60!)
    - rerank_depth=1000 gives best results
    
    Args:
        query: Query text
        rerank_depth: Number of docs to rerank (default: 1000)
        k_rrf: RRF constant (default: 30)
    
    Returns:
        List of (docid, score, rank) tuples
    """
    # Stage 1: Get BM25 and RM3 rankings
    bm25_results, rm3_results = multi_method_retrieval(query)
    
    # Build ranking dictionaries
    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}
    
    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        top_k_docids = [docid for docid, _ in bm25_results[:rerank_depth]]
        monot5_scores = rerank_monot5(query, top_k_docids)
        
        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}
    
    # Stage 3: WEIGHTED RRF Fusion
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys())
    rrf_scores = {}
    
    for docid in all_docids:
        rrf_score = 0.0
        
        # BM25 contribution (weight 1.0)
        if docid in bm25_ranking:
            rrf_score += W_BM25 * (1.0 / (k_rrf + bm25_ranking[docid]))
        
        # RM3 contribution (weight 1.5)
        if docid in rm3_ranking:
            rrf_score += W_RM3 * (1.0 / (k_rrf + rm3_ranking[docid]))
        
        # MonoT5 contribution (weight 2.0)
        if docid in monot5_ranking:
            rrf_score += W_MONOT5 * (1.0 / (k_rrf + monot5_ranking[docid]))
        
        rrf_scores[docid] = rrf_score
    
    # Sort by RRF score
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))
    
    return results

## Evaluate on Training Set

In [ ]:
print("="*70)
print("📊 EVALUATING HYBRID-OPTIMIZED PIPELINE")
print("="*70)
print("Configuration:")
print(f"  • Weighted RRF: W_BM25={W_BM25}, W_RM3={W_RM3}, W_MONOT5={W_MONOT5}")
print(f"  • k = {OPTIMAL_K}")
print(f"  • rerank_depth = {OPTIMAL_RERANK_DEPTH}")
print("="*70)
print()

import time

train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = hybrid_optimized_pipeline(query_text)

        for docid, score, rank in results:
            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  ✓ Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("📈 COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("🏆 TRAINING SET PERFORMANCE")
print("="*70)
print("\n📊 Your Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  ← Main metric")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

# Compare to tuning results
tuning_best = 0.2728  # k=30, depth=1000, pure RRF
original_weighted = 0.2738  # Original weighted RRF

print(f"\n📈 Comparison:")
print(f"  Original weighted RRF: {original_weighted:.4f}")
print(f"  Tuning best (pure RRF): {tuning_best:.4f}")
print(f"  This run (hybrid): {results_metrics[MAP]:.4f}")

if results_metrics[MAP] >= original_weighted:
    improvement = results_metrics[MAP] - original_weighted
    print(f"  ✓ Improvement: +{improvement:.4f} ({improvement/original_weighted*100:+.2f}%)")
else:
    decline = original_weighted - results_metrics[MAP]
    print(f"  ⚠️  Decline: -{decline:.4f} ({-decline/original_weighted*100:.2f}%)")

print(f"\n⏱️  Time: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

# Gap to 0.35
target = 0.35
gap = target - results_metrics[MAP]
print(f"\n🎯 Gap to MAP 0.35:")
print(f"  Current: {results_metrics[MAP]:.4f}")
print(f"  Target: {target:.4f}")
print(f"  Gap: {gap:.4f} ({gap/results_metrics[MAP]*100:.1f}% improvement needed)")

print("\n" + "="*70)
print("💡 TO REACH MAP 0.35:")
print("="*70)
print("Need to add more retrieval signals:")
print("  1. SPLADE (sparse learned retrieval)")
print("  2. Dense retrieval (ANCE, ColBERT)")
print("  3. Better reranker (MonoT5-3B, Cross-encoder)")
print("  4. Query expansion (Doc2Query, GPT-based)")
print("="*70)

## Generate Test Set Results

In [ ]:
import time

print("="*70)
print("🚀 GENERATING TEST RESULTS - HYBRID-OPTIMIZED")
print("="*70)
print(f"Configuration:")
print(f"  • Weighted RRF: W_BM25={W_BM25}, W_RM3={W_RM3}, W_MONOT5={W_MONOT5}")
print(f"  • k = {OPTIMAL_K}")
print(f"  • rerank_depth = {OPTIMAL_RERANK_DEPTH}")
print(f"  • Total test queries: {len(test_qids)}")
print("="*70)
print()

run3_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()
    results = hybrid_optimized_pipeline(query_text)
    query_time = time.time() - query_start

    for docid, score, rank in results:
        run3_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  ✓ Retrieved {len(results)} docs in {query_time:.1f}s")

    if i % 10 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("─"*70)
        print(f"📊 PROGRESS: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f} min, Remaining: {remaining/60:.1f} min")
        print(f"  Avg: {avg_time:.1f}s/query")
        print("─"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("✓ GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(run3_results):,}")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Avg: {total_time/len(test_qids):.1f}s/query")
print("="*70)

## Save Results

In [ ]:
with open('run_3_hybrid_optimized.res', 'w') as f:
    for r in run3_results:
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {r['score']:.6f} run3_hybrid_optimized\n")

print("✓ Saved to run_3_hybrid_optimized.res")
print("\nFirst 5 lines:")
with open('run_3_hybrid_optimized.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("📝 SUBMISSION READY!")
print("="*70)
print("File: run_3_hybrid_optimized.res")
print("\nKey optimizations applied:")
print("  ✓ Weighted RRF (proven better than pure RRF)")
print("  ✓ Optimal k=30 (from parameter tuning)")
print("  ✓ Optimal rerank_depth=1000 (from tuning)")
print("  ✓ Improved RM3 parameters")
print(f"\nExpected MAP: ~0.28-0.29 (baseline)")
print(f"To reach 0.35: Add SPLADE or dense retrieval")
print("="*70)